In [ ]:
from utils import *
import pickle

root_directory = "/nfs/datz/olmo_models/new_outputs/"

In [ ]:
with open(root_directory+"main/plays_sport.pkl", "rb") as file:
    ps_file = pickle.load(file)
with open(root_directory+"main/country_capital_city.pkl", "rb") as file:
    ccc_file = pickle.load(file)
with open(root_directory+"main/city_in_country.pkl", "rb") as file:
    cic_file = pickle.load(file)

In [ ]:
import os
import pickle

def load_and_filter(file_path):
    """Load a pickle file and filter entries with answer_token_span of length 1."""
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    return [entry for entry in data if len(entry.get("answer_token_span", [])) == 1]

# Define the root directory where all snapshots are stored.
root_directory = "/nfs/datz/olmo_models/new_outputs/"

# Choose one snapshot to load (e.g., "main")
reference_snapshot = "main"
reference_path = os.path.join(root_directory, reference_snapshot)

# Load and filter all relation files from the reference snapshot.
filtered_entries = {}
for file_name in os.listdir(reference_path):
    if file_name.endswith('.pkl'):
        stripped = file_name.split('.', 1)[0]
        file_path = os.path.join(reference_path, file_name)
        filtered_entries[stripped] = load_and_filter(file_path)

# Since the entries are the same in every snapshot, assign the filtered_entries to all snapshots.
all_snapshots_filtered = {}
for snapshot in os.listdir(root_directory):
    snapshot_path = os.path.join(root_directory, snapshot)
    if os.path.isdir(snapshot_path):
        all_snapshots_filtered[snapshot] = filtered_entries

# Example summary printout:
for snapshot, files in all_snapshots_filtered.items():
    print(f"Snapshot: {snapshot}")
    for file_name, entries in files.items():
        print(f"  {file_name}: {len(entries)} filtered entries")

In [ ]:
# Save the filtered_entries dictionary into a new pickle file.
output_file = os.path.join("./", "filtered_entries.pkl")
with open(output_file, "wb") as f:
    pickle.dump(all_snapshots_filtered, f)

In [ ]:

def load_filter_and_reduce(file_path, limit=5):
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    # Filter for entries with answer_token_span length equal to 1 and reduce to the first 2 entries.
    filtered = [entry for entry in data if len(entry.get("answer_token_span", [])) == 1]
    return filtered[:limit]

# Define the root directory where all snapshots are stored.
root_directory = "/nfs/datz/olmo_models/new_outputs/"

# Choose a reference snapshot (e.g., "main") to load the reduced entries from.
# reference_snapshot = "main"
# reference_path = os.path.join(root_directory, reference_snapshot)

# # Build a reduced dictionary with only 2 entries per relation from the reference snapshot.
# reduced_entries = {}
# for file_name in os.listdir(reference_path):
#     if file_name.endswith('.pkl'):
#         stripped = file_name.split('.', 1)[0]
#         file_path = os.path.join(reference_path, file_name)
#         reduced_entries[stripped] = load_filter_and_reduce(file_path, limit=2)

# Since the entries are identical across snapshots, assign the same reduced_entries to all snapshots.
all_snapshots_reduced = {}
for snapshot in os.listdir(root_directory):
    snapshot_path = os.path.join(root_directory, snapshot)
    if os.path.isdir(snapshot_path):
        reduced_entries = {}
        for file_name in os.listdir(snapshot_path):
            if file_name.endswith('.pkl'):
                stripped = file_name.split('.', 1)[0]
                file_path = os.path.join(snapshot_path, file_name)
                reduced_entries[stripped] = load_filter_and_reduce(file_path, limit=5)
    
        all_snapshots_reduced[snapshot] = reduced_entries
        
# Combine both loops into one dictionary comprehension:
# all_snapshots_reduced = {
#     snapshot: {
#         file_name.split('.', 1)[0]: load_filter_and_reduce(os.path.join(reference_path, file_name), limit=2)
#         for file_name in os.listdir(reference_path) if file_name.endswith('.pkl')
#     }
#     for snapshot in os.listdir(root_directory)
#     if os.path.isdir(os.path.join(root_directory, snapshot))
# }

# Save the combined reduced data to a pickle file.
output_file = os.path.join('./', "all_snapshots_reduced.pkl")
with open(output_file, "wb") as f:
    pickle.dump(all_snapshots_reduced, f)

print(f"Reduced entries for all snapshots saved to {output_file}")

In [ ]:
output_file = os.path.join('./', "all_snapshots_reduced.pkl")
with open(output_file, "rb") as f:
    all_snapshots_reduced = pickle.load(f)

In [ ]:
all_snapshots_reduced['main']['country_capital_city'][0]['full_graph'].number_of_nodes()
all_snapshots_reduced['main']['country_capital_city'][0]['full_graph'].number_of_edges()


In [ ]:
# Example: Assume your token subgraphs are stored in a list:
token_subgraphs= all_snapshots_reduced['main']['country_capital_city'][0]["token_subgraphs"]

# Choose one graph as the main graph, e.g. the first graph in the list.
main_graph = token_subgraphs[-2]

# The other graphs from which to subtract edges.
graphs_to_subtract = token_subgraphs[0:-2]

def subtract_graphs(G, H):
    """
    Returns a copy of G with all edges that appear in H removed.
    """
    result = G.copy()
    for u, v in H.edges():
        if result.has_edge(u, v):
            result.remove_edge(u, v)
    return result

# Start with a copy of the main graph and subtract edges from each other graph.
result_graph = main_graph.copy()
for subgraph in graphs_to_subtract:
    result_graph = subtract_graphs(result_graph, subgraph)

In [ ]:
for idx, subgraph in enumerate(token_subgraphs):
    # Assuming subgraph is a NetworkX graph, get the number of nodes and edges.
    num_nodes = len(subgraph.nodes())
    num_edges = len(subgraph.edges())
    print(f"  Subgraph {idx}: {num_nodes} nodes, {num_edges} edges")

In [ ]:
print(result_graph.number_of_nodes())
print(result_graph.number_of_edges())
print(token_subgraphs[1].nodes())

In [ ]:
import networkx as nx

# Example: Assume your token subgraphs are stored in a list:
token_subgraphs = all_snapshots_reduced['main']['country_capital_city'][0]["token_subgraphs"]

# Choose one graph as the main graph, e.g. the first graph in the list.
main_graph = token_subgraphs[-2]

# The other graphs from which to subtract edges.
graphs_to_subtract = token_subgraphs[0:-2]

def subtract_nodes_and_edges(G, H):
    """
    Subtract nodes and edges of graph H from graph G.
    This function removes:
      1. Every node in H from G (which automatically removes any incident edges)
      2. Any edge present in H from G (for extra safety)
    """
    result = G.copy()
    # Remove nodes that are in H
    nodes_to_remove = [node for node in H.nodes() if node in result]
    result.remove_nodes_from(nodes_to_remove)
    
    # Remove any remaining edges that appear in H (in case some nodes were not removed)
    for u, v in H.edges():
        if result.has_edge(u, v):
            result.remove_edge(u, v)
    
    return result


result_graph = main_graph.copy()
for subgraph in graphs_to_subtract:
    result_graph = subtract_nodes_and_edges(result_graph, subgraph)

In [ ]:
# Example: Assume your token subgraphs are stored in a list:
token_subgraphs2 = all_snapshots_reduced['main']['country_capital_city'][1]["token_subgraphs"]

# Choose one graph as the main graph, e.g. the first graph in the list.
main_graph2 = token_subgraphs2[-2]

# The other graphs from which to subtract edges.
graphs_to_subtract2 = token_subgraphs2[0:-2]

result_graph2 = main_graph.copy()
for subgraph in graphs_to_subtract2:
    result_graph2 = subtract_nodes_and_edges(result_graph2, subgraph)

In [ ]:
print(f"Nodes: {main_graph.number_of_nodes()}, Edges: {main_graph.number_of_edges()}")
print(f"Nodes: {main_graph2.number_of_nodes()}, Edges: {main_graph2.number_of_edges()}")

print(f"Nodes: {result_graph.number_of_nodes()}, Edges: {result_graph.number_of_edges()}")
print(f"Nodes: {result_graph2.number_of_nodes()}, Edges: {result_graph2.number_of_edges()}")

In [ ]:
all_snapshots_reduced['main']['country_capital_city'][0]

In [ ]:
for snapshot, relations in all_snapshots_reduced.items():
    print(f"### Snapshot: {snapshot} ###")
    for relation, entries in relations.items():
        print(f"\n--- Relation: {relation} ---")
        if len(entries) >= 2:
            # Get the second entry (index 1) from this relation.
            entry = entries[1]
            print("Sentence:", entry.get('sentence'))
            print("Tokens:", entry.get('tokens'))
            
            # Print the sizes of each token subgraph (if available)
            token_subgraphs = entry.get('token_subgraphs', [])
            if token_subgraphs:
                print("Token subgraphs sizes:")
                for idx, subgraph in enumerate(token_subgraphs):
                    try:
                        # Assuming subgraph is a NetworkX graph, get the number of nodes and edges.
                        num_nodes = len(subgraph.nodes())
                        num_edges = len(subgraph.edges())
                        print(f"  Subgraph {idx}: {num_nodes} nodes, {num_edges} edges")
                    except AttributeError:
                        # In case subgraph doesn't have nodes()/edges() methods.
                        print(f"  Subgraph {idx}: Cannot determine size (invalid graph object)")
            else:
                print("No token_subgraphs found in this entry.")
        else:
            print(f"The {relation} relation does not have a second entry.")

In [ ]:
import os
import pickle
import matplotlib.pyplot as plt

def snapshot_sort_key(snapshot_name):
    """Sort snapshots by step number; 'main' is considered the final snapshot."""
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0

# Specify the relation you want to analyze.
relation_to_analyze = "company_hq"  # Change as needed.

# Assume all_snapshots_reduced is already loaded.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

# Sort snapshots by their step number (with "main" coming last).
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# Lists to store snapshot labels and counts for both entries.
snapshots_list = []
nodes_entry0 = []
edges_entry0 = []
nodes_entry1 = []
edges_entry1 = []

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced[snapshot]
    if relation_to_analyze in relations:
        entries = relations[relation_to_analyze]
        if len(entries) >= 2:
            # For both entries, get the second-to-last token subgraph.
            entry0 = entries[0]
            entry1 = entries[1]
            token_subgraphs0 = entry0.get('token_subgraphs', [])
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            if len(token_subgraphs0) >= 2 and len(token_subgraphs1) >= 2:
                graph0 = token_subgraphs0[-2]
                graph1 = token_subgraphs1[-2]
                try:
                    n0 = len(graph0.nodes())
                    e0 = len(graph0.edges())
                    n1 = len(graph1.nodes())
                    e1 = len(graph1.edges())
                    
                    snapshots_list.append(snapshot)
                    nodes_entry0.append(n0)
                    edges_entry0.append(e0)
                    nodes_entry1.append(n1)
                    edges_entry1.append(e1)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both entries.")
        else:
            print(f"Snapshot '{snapshot}' does not have 2 entries for relation '{relation_to_analyze}'.")
    else:
        print(f"Relation '{relation_to_analyze}' not found in snapshot '{snapshot}'.")

# Plot a single line graph with four lines.
plt.figure(figsize=(12,6))
plt.plot(snapshots_list, nodes_entry0, marker='o', linestyle='-', label='Entry0 Nodes')
plt.plot(snapshots_list, edges_entry0, marker='o', linestyle='--', label='Entry0 Edges')
plt.plot(snapshots_list, nodes_entry1, marker='s', linestyle='-', label='Entry1 Nodes')
plt.plot(snapshots_list, edges_entry1, marker='s', linestyle='--', label='Entry1 Edges')
plt.xlabel("Snapshot")
plt.ylabel("Count")
plt.title(f"Counts for token_subgraphs[-2] for '{relation_to_analyze}'")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Specify the two relations to compare.
relation1 = "official_language"
relation2 = "country_capital_city"

# Load your all_snapshots_reduced dictionary.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

# Sort snapshots by their step number (with "main" coming last).
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

snapshots_list = []
jaccard_similarities = []

# Loop over each sorted snapshot.
for snapshot in sorted_snapshots:
    snapshot_data = all_snapshots_reduced[snapshot]
    if relation1 in snapshot_data and relation2 in snapshot_data:
        entries1 = snapshot_data[relation1]
        entries2 = snapshot_data[relation2]
        if len(entries1) >= 1 and len(entries2) >= 1:
            # Take one fact (the first entry) from each relation.
            fact1 = entries1[0]
            fact2 = entries2[0]
            token_subgraphs1 = fact1.get('token_subgraphs', [])
            token_subgraphs2 = fact2.get('token_subgraphs', [])
            if len(token_subgraphs1) >= 2 and len(token_subgraphs2) >= 2:
                # Use the second-to-last token subgraph for each fact.
                graph1 = token_subgraphs1[-2]
                graph2 = token_subgraphs2[-2]
                try:
                    sim = jaccard_similarity(graph1, graph2)
                    snapshots_list.append(snapshot)
                    jaccard_similarities.append(sim)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both relations.")
        else:
            print(f"Snapshot '{snapshot}' does not have at least one entry for both relations.")
    else:
        print(f"Snapshot '{snapshot}' is missing one of the relations.")

# Plot the Jaccard similarities over snapshots as a single line graph.
plt.figure(figsize=(10, 6))
plt.plot(snapshots_list, jaccard_similarities, marker='o', linestyle='-')
plt.xlabel("Snapshot")
plt.ylabel("Jaccard Similarity")
plt.title(f"Jaccard Similarity between {relation1} and {relation2} (token_subgraphs[-2])")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Specify the two relations to compare.
relation1 = "official_language"
relation2 = "company_hq"

# Assume all_snapshots_reduced is already loaded.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

# Sort snapshots by their step number (with "main" coming last).
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# Initialize lists to store snapshot labels and the total counts for each relation.
snapshots_list = []
relation1_total = []
relation2_total = []

for snapshot in sorted_snapshots:
    snapshot_data = all_snapshots_reduced[snapshot]
    if relation1 in snapshot_data and relation2 in snapshot_data:
        entries1 = snapshot_data[relation1]
        entries2 = snapshot_data[relation2]
        if len(entries1) >= 1 and len(entries2) >= 1:
            # Take one fact (the first entry) from each relation.
            fact1 = entries1[0]
            fact2 = entries2[0]
            token_subgraphs1 = fact1.get('token_subgraphs', [])
            token_subgraphs2 = fact2.get('token_subgraphs', [])
            if len(token_subgraphs1) >= 2 and len(token_subgraphs2) >= 2:
                # Use the second-to-last token subgraph from each fact.
                graph1 = token_subgraphs1[-2]
                graph2 = token_subgraphs2[-2]
                try:
                    n1 = len(graph1.nodes())
                    e1 = len(graph1.edges())
                    n2 = len(graph2.nodes())
                    e2 = len(graph2.edges())
                    
                    total1 = n1 + e1
                    total2 = n2 + e2
                    
                    snapshots_list.append(snapshot)
                    relation1_total.append(total1)
                    relation2_total.append(total2)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both relations.")
        else:
            print(f"Snapshot '{snapshot}' does not have at least one entry for both relations.")
    else:
        print(f"Snapshot '{snapshot}' is missing one of the relations.")

# Plot the total counts over snapshots as a single line graph.
plt.figure(figsize=(12, 6))
plt.plot(snapshots_list, relation1_total, marker='o', linestyle='-', label=f"{relation1} Total (Nodes+Edges)")
plt.plot(snapshots_list, relation2_total, marker='s', linestyle='-', label=f"{relation2} Total (Nodes+Edges)")
plt.xlabel("Snapshot")
plt.ylabel("Total Count (Nodes + Edges)")
plt.title("Evolution of Total Counts for token_subgraphs[-2]")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def graph_signature(G):
    """
    Create a signature for a graph by tagging nodes and edges.
    Returns a set containing tuples for nodes and edges.
    """
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def jaccard_similarity(G1, G2):
    """
    Compute the Jaccard similarity between two graphs based on their signatures.
    """
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

def snapshot_sort_key(snapshot_name):
    """Sort snapshots by step number; 'main' is considered the final snapshot."""
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0

# Specify the relation to analyze.
relation_to_analyze = "city_in_country"  # Adjust if needed.

# Load your all_snapshots_reduced dictionary.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

# Sort the snapshot names.
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# Lists to store snapshots and the corresponding Jaccard similarity.
snapshots_list = []
jaccard_similarities = []

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced[snapshot]
    if relation_to_analyze in relations:
        entries = relations[relation_to_analyze]
        if len(entries) >= 2:
            entry0 = entries[0]
            entry1 = entries[1]
            token_subgraphs0 = entry0.get('token_subgraphs', [])
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            if len(token_subgraphs0) >= 2 and len(token_subgraphs1) >= 2:
                # Use the second-to-last token subgraph.
                graph0 = token_subgraphs0[entry0.get('answer_token_span')[0]-1]
                graph1 = token_subgraphs1[entry1.get('answer_token_span')[0]-1]
                try:
                    sim = jaccard_similarity(graph0, graph1)
                    snapshots_list.append(snapshot)
                    jaccard_similarities.append(sim)
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both entries.")
        else:
            print(f"Snapshot '{snapshot}' does not have 2 entries for relation '{relation_to_analyze}'.")
    else:
        print(f"Relation '{relation_to_analyze}' not found in snapshot '{snapshot}'.")

# Plot the Jaccard similarities over snapshots as one line graph.
plt.figure(figsize=(10, 6))
plt.plot(snapshots_list, jaccard_similarities, marker='o', linestyle='-')
plt.xlabel("Snapshot")
plt.ylabel("Jaccard Similarity")
plt.title(f"Jaccard Similarity of token_subgraphs for '{relation_to_analyze}'")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx
import pickle
import matplotlib.pyplot as plt

def graph_signature(G):
    """
    Create a signature for a graph by tagging nodes and edges.
    Returns a set containing tuples for nodes and edges.
    """
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def jaccard_similarity(G1, G2):
    """
    Compute the Jaccard similarity between two graphs based on their signatures.
    """
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

def graph_diff(G1, G2):
    """
    Compute differences between two graphs.
    Returns:
      - common_nodes: nodes present in both graphs.
      - added_nodes: nodes that are in G2 but not in G1.
      - removed_nodes: nodes that are in G1 but not in G2.
      - common_edges: edges present in both graphs.
      - added_edges: edges that are in G2 but not in G1.
      - removed_edges: edges that are in G1 but not in G2.
    """
    nodes1 = {("node", n) for n in G1.nodes()}
    nodes2 = {("node", n) for n in G2.nodes()}
    edges1 = {("edge", u, v) for u, v in G1.edges()}
    edges2 = {("edge", u, v) for u, v in G2.edges()}

    common_nodes = nodes1.intersection(nodes2)
    added_nodes = nodes2 - nodes1
    removed_nodes = nodes1 - nodes2

    common_edges = edges1.intersection(edges2)
    added_edges = edges2 - edges1
    removed_edges = edges1 - edges2

    return common_nodes, added_nodes, removed_nodes, common_edges, added_edges, removed_edges

def snapshot_sort_key(snapshot_name):
    """
    Sort snapshots by step number; 'main' is considered the final snapshot.
    """
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0

# Load your all_snapshots_reduced dictionary.
# For example, if your file is "all_snapshots_reduced.pkl":
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

relation_to_analyze = "official_language"  # Adjust if needed.

# Sort the snapshot names.
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# We'll compare the same entry across consecutive snapshots.
# Choose a fixed entry index. Here we use entry index 0.
entry_index = 0

snapshots_pairs = []       # To store the snapshot pair names (e.g., "snap1 -> snap2")
jaccard_similarities = []  # To store computed Jaccard similarities for the pairs

for i in range(1, len(sorted_snapshots)):
    prev_snapshot = sorted_snapshots[i-1]
    curr_snapshot = sorted_snapshots[i]
    
    prev_relations = all_snapshots_reduced.get(prev_snapshot, {})
    curr_relations = all_snapshots_reduced.get(curr_snapshot, {})
    
    if relation_to_analyze in prev_relations and relation_to_analyze in curr_relations:
        prev_entries = prev_relations[relation_to_analyze]
        curr_entries = curr_relations[relation_to_analyze]
        
        if len(prev_entries) > entry_index and len(curr_entries) > entry_index:
            entry_prev = prev_entries[entry_index]
            entry_curr = curr_entries[entry_index]
            
            token_subgraphs_prev = entry_prev.get('token_subgraphs', [])
            token_subgraphs_curr = entry_curr.get('token_subgraphs', [])
            
            if len(token_subgraphs_prev) >= 2 and len(token_subgraphs_curr) >= 2:
                # Use the second-to-last token subgraph from each snapshot.
                graph_prev = token_subgraphs_prev[-2]
                graph_curr = token_subgraphs_curr[-2]
                
                try:
                    # Calculate Jaccard similarity.
                    sim = jaccard_similarity(graph_prev, graph_curr)
                    jaccard_similarities.append(sim)
                    snapshots_pairs.append(f"{prev_snapshot} -> {curr_snapshot}")
                    
                    # Calculate total number of nodes and edges.
                    total_nodes_prev = graph_prev.number_of_nodes()
                    total_edges_prev = graph_prev.number_of_edges()
                    total_nodes_curr = graph_curr.number_of_nodes()
                    total_edges_curr = graph_curr.number_of_edges()
                    
                    # Calculate differences.
                    common_nodes, added_nodes, removed_nodes, common_edges, added_edges, removed_edges = graph_diff(graph_prev, graph_curr)
                    
                    print(f"Comparing snapshots: {prev_snapshot} -> {curr_snapshot}")
                    print(f"Jaccard Similarity: {sim:.3f}")
                    print(f"Snapshot {prev_snapshot}: {total_nodes_prev} nodes, {total_edges_prev} edges")
                    print(f"Snapshot {curr_snapshot}: {total_nodes_curr} nodes, {total_edges_curr} edges")
                    print(f"Common nodes ({len(common_nodes)}): {common_nodes}")
                    print(f"Added nodes ({len(added_nodes)}): {added_nodes}")
                    print(f"Removed nodes ({len(removed_nodes)}): {removed_nodes}")
                    # print(f"Common edges ({len(common_edges)}): {common_edges}")
                    # print(f"Added edges ({len(added_edges)}): {added_edges}")
                    # print(f"Removed edges ({len(removed_edges)}): {removed_edges}")
                    print("="*50)
                except Exception as err:
                    print(f"Error comparing snapshots {prev_snapshot} and {curr_snapshot}: {err}")
            else:
                print(f"Snapshot pair {prev_snapshot} and {curr_snapshot} do not have at least 2 token_subgraphs in the chosen entry.")
        else:
            print(f"Snapshot pair {prev_snapshot} and {curr_snapshot} do not have entry index {entry_index} for relation '{relation_to_analyze}'.")
    else:
        print(f"Relation '{relation_to_analyze}' not found in one of the snapshots: {prev_snapshot} or {curr_snapshot}.")

# Plot the Jaccard similarities over snapshot pairs.
if snapshots_pairs:
    plt.figure(figsize=(12, 6))
    plt.plot(snapshots_pairs, jaccard_similarities, marker='o', linestyle='-')
    plt.xlabel("Snapshot Pair")
    plt.ylabel("Jaccard Similarity")
    plt.title(f"Jaccard Similarity for '{relation_to_analyze}' (Entry index {entry_index}) across Snapshots")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No valid snapshot pairs found to compute differences.")


In [ ]:
import networkx as nx
import pickle
import matplotlib.pyplot as plt

def graph_signature(G):
    """
    Create a signature for a graph by tagging nodes and edges.
    Returns a set containing tuples for nodes and edges.
    """
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def snapshot_sort_key(snapshot_name):
    """
    Sort snapshots by step number; 'main' is considered the final snapshot.
    """
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0

# Load your all_snapshots_reduced dictionary.
# For example:
# with open("all_snapshots_reduced.pkl", "rb") as f:
#     all_snapshots_reduced = pickle.load(f)

relation_to_analyze = "official_language"  # Adjust if needed.
entry_index = 0  # Choose which entry to track.
token_subgraph_index = -2  # Using the second-to-last token subgraph as before.

# Sort the snapshot names.
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

persistent_nodes = None
persistent_edges = None

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced.get(snapshot, {})
    if relation_to_analyze not in relations:
        print(f"Relation '{relation_to_analyze}' not found in snapshot '{snapshot}'. Skipping...")
        continue

    entries = relations[relation_to_analyze]
    if len(entries) <= entry_index:
        print(f"Snapshot '{snapshot}' does not have entry index {entry_index} for relation '{relation_to_analyze}'. Skipping...")
        continue

    entry = entries[entry_index]
    token_subgraphs = entry.get('token_subgraphs', [])
    if len(token_subgraphs) < abs(token_subgraph_index):
        print(f"Snapshot '{snapshot}' does not have at least {abs(token_subgraph_index)} token_subgraphs. Skipping...")
        continue

    # Get the selected graph.
    graph_current = token_subgraphs[token_subgraph_index]

    # Build the sets of nodes and edges for the current graph.
    nodes_current = {("node", n) for n in graph_current.nodes()}
    edges_current = {("edge", u, v) for u, v in graph_current.edges()}

    if persistent_nodes is None:
        # Initialize persistent sets on the first valid snapshot.
        persistent_nodes = nodes_current
        persistent_edges = edges_current
    else:
        # Intersect with the persistent sets from previous snapshots.
        persistent_nodes = persistent_nodes.intersection(nodes_current)
        persistent_edges = persistent_edges.intersection(edges_current)

# Output the nodes and edges that are always part of the circuit.
print("Nodes that are never removed (persistent across all snapshots):")
print(persistent_nodes)
print("\nEdges that are never removed (persistent across all snapshots):")
print(persistent_edges)

In [ ]:
#ps_file = reduced_entries.get("plays_sport", [])

for y in reduced_entries.items():
# Check if there is at least a second entry
    for x in y[1]:
        if len(x) > 1:
            entry = x
            print("Sentence:", entry.get('sentence'))
            print("Tokens:", entry.get('tokens'))
            
            # Check if 'token_subgraphs' exists and print each subgraph
            token_subgraphs = entry.get('token_subgraphs', [])
            if token_subgraphs:
                print("Token subgraphs:")
                for idx, subgraph in enumerate(token_subgraphs):
                    print(f"  Subgraph {idx}:", subgraph)
            else:
                print("No token_subgraphs found in this entry.")
        else:
            print("The plays_sport relation does not have a second entry.")

In [ ]:
import os
import pickle
import networkx as nx

def graph_signature(G):
    """
    Create a signature for a graph by creating sets for nodes and edges
    with different tags to avoid collisions.
    """
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def jaccard_similarity(G1, G2):
    """
    Compute the Jaccard similarity between two graphs based on their signatures.
    """
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

# Load the all_snapshots_reduced pickle file.
with open("all_snapshots_reduced.pkl", "rb") as f:
    all_snapshots_reduced = pickle.load(f)

# Get a sorted list of snapshot names.
snapshots = sorted(all_snapshots_reduced.keys())
print("Comparing revisions (snapshots) from revision to revision:")

# Loop pairwise through the sorted snapshots.
for i in range(len(snapshots) - 1):
    snap1 = snapshots[i]
    snap2 = snapshots[i + 1]
    print(f"\n--- Comparing snapshot '{snap1}' with snapshot '{snap2}' ---")
    
    # For each relation in the snapshot
    for relation in all_snapshots_reduced[snap1]:
        entries1 = all_snapshots_reduced[snap1][relation]
        entries2 = all_snapshots_reduced[snap2][relation]
        
        # Ensure both snapshots have at least 2 entries for this relation.
        if len(entries1) >= 2 and len(entries2) >= 2:
            entry1 = entries1[1]  # second entry of snap1
            entry2 = entries2[1]  # second entry of snap2
            
            # Print sentences from both revisions.
            print(f"Relation: {relation}")
            print(f"  {snap1} sentence: {entry1.get('sentence')}")
            print(f"  {snap2} sentence: {entry2.get('sentence')}")
            
            # Compare the last token_subgraph of each entry.
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            token_subgraphs2 = entry2.get('token_subgraphs', [])
            if token_subgraphs1 and token_subgraphs2:
                G1 = token_subgraphs1[-1]
                G2 = token_subgraphs2[-1]
                similarity = jaccard_similarity(G1, G2)
                print(f"  Jaccard similarity of last token subgraph: {similarity}")
            else:
                print("  Missing token_subgraphs in one of the entries.")
        else:
            print(f"Relation '{relation}' does not have enough entries in one of the snapshots.")

In [ ]:
import os
import pickle

def graph_signature(G):
    # Create a signature that tags nodes and edges differently to avoid collisions.
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

def jaccard_similarity(G1, G2):
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

# # Load the reduced entries pickle file
# root_directory = "/nfs/datz/olmo_models/new_outputs/"
# with open(os.path.join(root_directory, "reduced_entries.pkl"), "rb") as f:
#     reduced_entries = pickle.load(f)

print("Within-relation Jaccard similarities (using last token_subgraph):")
for relation, entries in reduced_entries.items():
    if len(entries) >= 2:
        # Retrieve the list of token_subgraphs for each entry.
        subgraphs_1 = entries[0].get('token_subgraphs', [])
        subgraphs_2 = entries[1].get('token_subgraphs', [])
        if subgraphs_1 and subgraphs_2:
            # Use the last token subgraph from each list.
            G1 = subgraphs_1[-2]
            G2 = subgraphs_2[-2]
            sim = jaccard_similarity(G1, G2)
            print(f"  {relation}: {sim}")
        else:
            print(f"  {relation}: Missing token_subgraphs in one or both entries.")
    else:
        print(f"  {relation}: Not enough entries for within-relation comparison.")

print("\nCross-relation Jaccard similarities (using last token_subgraph of the first entry):")
relations = list(reduced_entries.keys())
for i in range(len(relations)):
    for j in range(i + 1, len(relations)):
        rel1 = relations[i]
        rel2 = relations[j]
        entries1 = reduced_entries[rel1]
        entries2 = reduced_entries[rel2]
        if entries1 and entries2:
            subgraphs_1 = entries1[0].get('token_subgraphs', [])
            subgraphs_2 = entries2[0].get('token_subgraphs', [])
            if subgraphs_1 and subgraphs_2:
                # Use the last token subgraph from the first entry of each relation.
                G1 = subgraphs_1[-2]
                G2 = subgraphs_2[-2]
                sim = jaccard_similarity(G1, G2)
                print(f"  {rel1} vs {rel2}: {sim}")
            else:
                print(f"  {rel1} vs {rel2}: Missing token_subgraphs in one of the entries.")

In [ ]:
print(ps_file[1]['sentence'])
print(ps_file[1]['tokens'])
for x in ps_file[1]['token_subgraphs']:
    print(x)

In [ ]:
print(ccc_file[7]['sentence'])
#print(ccc_file[7]['tokens'])
t = len(ccc_file[8]['tokens'])
#for x in ccc_file[7]['token_subgraphs']:
print(ccc_file[7]['token_subgraphs'][t-2])

In [ ]:
import networkx as nx

# Assuming these objects are DiGraphs from NetworkX
t_ccc = len(ccc_file[7]['tokens'])
t_ps = len(ps_file[1]['tokens'])
ccc_graph = ccc_file[7]['token_subgraphs'][t_ccc-2]
ps_graph = ps_file[1]['token_subgraphs'][t_ps-2]

t_cic = len(cic_file[0]['tokens'])
cic_graph = cic_file[0]['token_subgraphs'][t_cic-2]


def graph_signature(G):
    # Create a signature that tags nodes and edges differently to avoid collisions.
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

# Create the signatures for each graph
sig_ccc = graph_signature(ccc_graph)
sig_ps = graph_signature(ps_graph)

# Compute the intersection and union of the signatures
intersection = sig_ccc.intersection(sig_ps)
union = sig_ccc.union(sig_ps)

# Calculate the Jaccard similarity
jaccard_joint = len(intersection) / len(union) if union else 1.0
print("Joint Jaccard similarity (nodes and edges):", jaccard_joint)

In [ ]:
sig_ccc = graph_signature(ccc_graph)
sig_cic = graph_signature(cic_graph)

# Compute the intersection and union of the signatures
intersection = sig_ccc.intersection(sig_cic)
union = sig_ccc.union(sig_cic)

# Calculate the Jaccard similarity
jaccard_joint = len(intersection) / len(union) if union else 1.0
print("Joint Jaccard similarity (nodes and edges):", jaccard_joint)

In [ ]:
t_ccc = len(ccc_file[7]['tokens'])
ccc_graph = ccc_file[7]['token_subgraphs'][t_ccc-2]

t_ccc_2 = len(ccc_file[3]['tokens'])
ccc_graph_2 = ccc_file[3]['token_subgraphs'][t_ccc_2-2]


# Create the signatures for each graph
sig_ccc = graph_signature(ccc_graph)
sig_ps = graph_signature(ccc_graph_2)

# Compute the intersection and union of the signatures
intersection = sig_ccc.intersection(sig_ps)
union = sig_ccc.union(sig_ps)

# Calculate the Jaccard similarity
jaccard_joint = len(intersection) / len(union) if union else 1.0
print("Joint Jaccard similarity (nodes and edges):", jaccard_joint)

In [ ]:
def get_output_node(n_layers, token: int):
    return f"I{n_layers - 1}_{token}"

def change_threshold_graph(graph, threshold):
    rgraph = graph.reverse()
    search_graph = nx.subgraph_view(
        rgraph, filter_edge=lambda u, v: rgraph[u][v]["weight"] > threshold
    )

    result = []
    for start in starting_tokens:
        assert start < n_tokens
        assert start >= 0
        edges = nx.edge_dfs(search_graph, source=get_output_node(32, start))
        tree = search_graph.edge_subgraph(edges)
        # Reverse the edges because the dfs was going from upper layer downwards.
        result.append(tree.reverse())

    return result



In [ ]:
import networkx as nx

def get_output_node(n_layers, token: int):
    return f"I{n_layers - 1}_{token}"

def change_threshold_graph_for_token(graph, threshold, answer_token_pos):
    rgraph = graph.reverse()
    search_graph = nx.subgraph_view(
        rgraph, filter_edge=lambda u, v: rgraph[u][v]["weight"] > threshold
    )
    # Get the node corresponding to the token in the last layer.
    token_node = get_output_node(32, answer_token_pos)
    edges = nx.edge_dfs(search_graph, source=token_node)
    tree = search_graph.edge_subgraph(edges)
    # Reverse the edges because the DFS was performed from the output layer downwards.
    return tree.reverse()

In [ ]:
all_snapshots_filtered['main']['country_capital_city'][4]['sentence']

In [ ]:
relation = "country_capital_city"
token_subgraphs1 = all_snapshots_filtered['main'][relation][2]["token_subgraphs"][-2]
token_subgraphs2 = all_snapshots_filtered['main'][relation][3]["token_subgraphs"][-2]

answer_token_pos1 = all_snapshots_filtered['main'][relation][2]["answer_token_span"][0] - 1
answer_token_pos2 = all_snapshots_filtered['main'][relation][3]["answer_token_span"][0] - 1

print(len(all_snapshots_filtered['main'][relation][2]["tokens"]))
print(len(all_snapshots_filtered['main'][relation][3]["tokens"]))


In [ ]:
all_snapshots_reduced['main']['country_capital_city'][1]

In [ ]:
print(len(all_snapshots_reduced['main']['company_hq'][0]["tokens"]))
print(all_snapshots_reduced['main']['company_hq'][4])

In [ ]:
new_graph1 = change_threshold_graph_for_token(token_subgraphs1, 0.03, answer_token_pos1)
new_graph2 = change_threshold_graph_for_token(token_subgraphs2, 0.03, answer_token_pos2)
print(f"Size of new_graph1 {new_graph1.number_of_nodes()}, {new_graph1.number_of_edges()}")
print(f"Size of new_graph1 {new_graph2.number_of_nodes()}, {new_graph2.number_of_edges()}")

In [ ]:
# Create the signatures for each graph
sig_1 = graph_signature(new_graph1)
sig_2 = graph_signature(new_graph2)

# Compute the intersection and union of the signatures
intersection = sig_1.intersection(sig_2)
union = sig_1.union(sig_2)

# Calculate the Jaccard similarity
jaccard_joint = len(intersection) / len(union) if union else 1.0
print("Joint Jaccard similarity (nodes and edges):", jaccard_joint)

In [ ]:
all_snapshots_reduced.keys()

In [ ]:
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

for snapshot in sorted_snapshots:
    # Access the plays_sport data for the current snapshot
    plays_data = all_snapshots_reduced[snapshot]['company_hq']
    
    # Extract the token subgraphs (using the second-to-last element in the list)
    token_subgraphs1 = plays_data[0]["token_subgraphs"][-2]
    token_subgraphs2 = plays_data[1]["token_subgraphs"][-2]
    
    # Extract the answer token positions (subtracting 1 from the first element of the span)
    answer_token_pos1 = plays_data[0]["answer_token_span"][0] - 1
    answer_token_pos2 = plays_data[1]["answer_token_span"][0] - 1
    
    # Create new graphs using your threshold function
    new_graph1 = change_threshold_graph_for_token(token_subgraphs1, 0.03, answer_token_pos1)
    new_graph2 = change_threshold_graph_for_token(token_subgraphs2, 0.03, answer_token_pos2)
    
    # Print the size of each graph for the current snapshot
    print(f"Snapshot: {snapshot}")
    print(f"Graph 1: Nodes = {new_graph1.number_of_nodes()}, Edges = {new_graph1.number_of_edges()}")
    print(f"Graph 2: Nodes = {new_graph2.number_of_nodes()}, Edges = {new_graph2.number_of_edges()}")
    
    # Create the signatures for each graph
    sig_1 = graph_signature(new_graph1)
    sig_2 = graph_signature(new_graph2)
    
    # Compute the intersection and union of the signatures
    intersection = sig_1.intersection(sig_2)
    union = sig_1.union(sig_2)
    
    # Calculate the Jaccard similarity
    jaccard_joint = len(intersection) / len(union) if union else 1.0
    print("Joint Jaccard similarity (nodes and edges):", jaccard_joint)
    print("=" * 40)


In [ ]:
import pickle
import matplotlib.pyplot as plt
import networkx as nx

def snapshot_sort_key(snapshot_name):
    """Sort snapshots by step number; 'main' is considered the final snapshot."""
    if snapshot_name == "main":
        return float('inf')
    try:
        # Extract the numeric part after 'step' before the first dash.
        step_str = snapshot_name.split("step")[1].split('-')[0]
        return int(step_str)
    except Exception:
        return 0

# Define the graph signature function.
def graph_signature(G):
    # Tags nodes and edges differently to avoid collisions.
    nodes = {("node", n) for n in G.nodes()}
    edges = {("edge", u, v) for u, v in G.edges()}
    return nodes.union(edges)

# Define the Jaccard similarity function.
def jaccard_similarity(G1, G2):
    sig1 = graph_signature(G1)
    sig2 = graph_signature(G2)
    intersection = sig1.intersection(sig2)
    union = sig1.union(sig2)
    return len(intersection) / len(union) if union else 1.0

# Specify the relation to analyze.
relation_to_analyze = "country_capital_city"

# Sort the snapshot names.
sorted_snapshots = sorted(all_snapshots_reduced.keys(), key=snapshot_sort_key)

# Lists to store snapshots and the corresponding metrics.
snapshots_list = []
jaccard_similarities = []
nodes_count0 = []
nodes_count1 = []
edges_count0 = []
edges_count1 = []

for snapshot in sorted_snapshots:
    relations = all_snapshots_reduced[snapshot]
    if relation_to_analyze in relations:
        entries = relations[relation_to_analyze]
        if len(entries) >= 2:
            entry0 = entries[2]
            entry1 = entries[3]
            token_subgraphs0 = entry0.get('token_subgraphs', [])
            token_subgraphs1 = entry1.get('token_subgraphs', [])
            if len(token_subgraphs0) >= 2 and len(token_subgraphs1) >= 2:
                # Use the second-to-last token subgraph.
                token_subgraph0 = token_subgraphs0[-2]
                token_subgraph1 = token_subgraphs1[-2]
                # Compute the answer token positions (subtract 1 as before).
                answer_token_pos0 = entry0.get('answer_token_span')[0] - 1
                answer_token_pos1 = entry1.get('answer_token_span')[0] - 1
                try:
                    # Apply threshold filtering.
                    graph0 = change_threshold_graph_for_token(token_subgraph0, 0.03, answer_token_pos0)
                    graph1 = change_threshold_graph_for_token(token_subgraph1, 0.03, answer_token_pos1)
                    
                    # Compute Jaccard similarity between the two graphs.
                    sim = jaccard_similarity(graph0, graph1)
                    snapshots_list.append(snapshot)
                    jaccard_similarities.append(sim)
                    
                    # Record the number of nodes and edges for each graph.
                    nodes_count0.append(graph0.number_of_nodes())
                    nodes_count1.append(graph1.number_of_nodes())
                    edges_count0.append(graph0.number_of_edges())
                    edges_count1.append(graph1.number_of_edges())
                except Exception as err:
                    print(f"Error processing snapshot '{snapshot}': {err}")
            else:
                print(f"Snapshot '{snapshot}' does not have at least 2 token_subgraphs for both entries.")
        else:
            print(f"Snapshot '{snapshot}' does not have 2 entries for relation '{relation_to_analyze}'.")
    else:
        print(f"Relation '{relation_to_analyze}' not found in snapshot '{snapshot}'.")

# Plot the Jaccard similarities over snapshots.
plt.figure(figsize=(10, 6))
plt.plot(snapshots_list, jaccard_similarities, marker='o', linestyle='-')
plt.xlabel("Snapshot")
plt.ylabel("Jaccard Similarity")
plt.title(f"Jaccard Similarity of threshold-filtered token_subgraphs for '{relation_to_analyze}'")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Plot the number of nodes and edges over snapshots in a separate figure.
plt.figure(figsize=(12, 8))

# Subplot for node counts.
plt.subplot(2, 1, 1)
plt.plot(snapshots_list, nodes_count0, marker='o', linestyle='-', label='Graph 0 Nodes')
plt.plot(snapshots_list, nodes_count1, marker='o', linestyle='-', label='Graph 1 Nodes')
plt.xlabel("Snapshot")
plt.ylabel("Number of Nodes")
plt.title("Number of Nodes over Snapshots")
plt.legend()
plt.xticks(rotation=90)

# Subplot for edge counts.
plt.subplot(2, 1, 2)
plt.plot(snapshots_list, edges_count0, marker='o', linestyle='-', label='Graph 0 Edges')
plt.plot(snapshots_list, edges_count1, marker='o', linestyle='-', label='Graph 1 Edges')
plt.xlabel("Snapshot")
plt.ylabel("Number of Edges")
plt.title("Number of Edges over Snapshots")
plt.legend()
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
import re

def get_token_index(node_name):
    """
    Extracts the token index from a node name.
    Example: 'A0_2' -> 0, 'I3_0' -> 3.
    """
    match = re.match(r'[A-Z]\d+_(\d+)', node_name)
    if match:
        return int(match.group(1))
    return None

def average_edge_weight_from_subtokens(edges, subject_span, answer_span):
    """
    Computes the average edge weight from all edges where:
      - the source node's token index is in subject_span, and
      - the target node's token index is in answer_span.
    
    edges: an iterable of tuples (src, tgt, data) where data is a dict with key 'weight'
    subject_span: list (or set) of token indices for the subject (e.g., [0, 1])
    answer_span: list (or set) of token indices for the answer (e.g., [7])
    """
    weights = []
    for src, tgt, data in edges:
        src_idx = get_token_index(src)
        tgt_idx = get_token_index(tgt)
        if src_idx in subject_span and tgt_idx in answer_span:
            weights.append(data['weight'])
    if weights:
        return sum(weights) / len(weights)
    else:
        return None  # or 0.0, or raise an error if no edges are found

In [ ]:
edges = list(all_snapshots_reduced['main']['country_capital_city'][4]['token_subgraphs'][-2].edges().data())
subject_span = all_snapshots_reduced['main']['country_capital_city'][4]['subj_token_span']
answer_span = [x - 1 for x in all_snapshots_reduced['main']['country_capital_city'][4]['answer_token_span']]

In [ ]:
all_snapshots_reduced['main']['country_capital_city'][4]['sentence']

In [ ]:
token_subgraph = all_snapshots_reduced['main']['country_capital_city'][1]
token_subgraph

In [ ]:
import networkx as nx

# Retrieve the token subgraph and spans from your dict.
token_subgraph = all_snapshots_reduced['main']['country_capital_city'][1]['token_subgraphs'][-2]
subject_span = all_snapshots_reduced['main']['country_capital_city'][1]['subj_token_span']
# Adjust answer token indices as you mentioned.
answer_span = [x - 1 for x in all_snapshots_reduced['main']['country_capital_city'][1]['answer_token_span']]

def extract_token_index(node_name: str) -> int:
    """
    Extracts the token index from a node name.
    Assumes node names are of the form 'I0_0', 'A1_3', etc.
    """
    return int(node_name.split('_')[-1])

# Dictionary to hold the list of weights for each answer node.
aggregated_edges = {}
# List to keep track of which edges to remove.
edges_to_remove = []

# Iterate through all edges in the token_subgraph.
for u, v, data in token_subgraph.edges(data=True):
    src_index = extract_token_index(u)
    dst_index = extract_token_index(v)
    # Check if the source is in the subject span and the destination is in the answer span.
    if src_index in subject_span and dst_index in answer_span:
        # Group by destination (answer node).
        aggregated_edges.setdefault(v, []).append(data['weight'])
        edges_to_remove.append((u, v))

# Remove all the original subject->answer edges.
for u, v in edges_to_remove:
    token_subgraph.remove_edge(u, v)

# Define a new node that will represent the aggregated subject.
aggregated_subject_node = "subject_aggregated"
if not token_subgraph.has_node(aggregated_subject_node):
    token_subgraph.add_node(aggregated_subject_node)

# For each answer node, add a single aggregated edge from the aggregated subject node.
for answer_node, weights in aggregated_edges.items():
    avg_weight = sum(weights) / len(weights)
    token_subgraph.add_edge(aggregated_subject_node, answer_node, weight=avg_weight)
    print(f"Aggregated edge: ({aggregated_subject_node}, {answer_node}) with average weight {avg_weight}")

In [ ]:
list(token_subgraph.edges())

In [ ]:
list(token_subgraph.edges())

In [ ]:
import networkx as nx

# Get the original token subgraph and token spans.
token_subgraph = all_snapshots_reduced['main']['country_capital_city'][4]['token_subgraphs'][-2]
subject_span = all_snapshots_reduced['main']['country_capital_city'][4]['subj_token_span']
answer_span = [x - 1 for x in all_snapshots_reduced['main']['country_capital_city'][4]['answer_token_span']]

def extract_token_index(node_name: str) -> int:
    """Extract the token index assuming node names like 'I0_0', 'A1_3', etc."""
    return int(node_name.split('_')[-1])

# Identify subject and answer nodes in the original subgraph.
subject_nodes = [node for node in token_subgraph.nodes() if extract_token_index(node) in subject_span]
answer_nodes = [node for node in token_subgraph.nodes() if extract_token_index(node) in answer_span]

# Create a new graph to hold the aggregated nodes.
aggregated_graph = nx.DiGraph()

# Add all nodes that are not in the subject or answer groups.
for node in token_subgraph.nodes():
    if node not in subject_nodes and node not in answer_nodes:
        aggregated_graph.add_node(node)

# Add the new aggregated nodes.
aggregated_subject = "subject_aggregated"
aggregated_answer = "answer_aggregated"
aggregated_graph.add_node(aggregated_subject)
aggregated_graph.add_node(aggregated_answer)

# We'll accumulate edges in a dictionary with keys (new_source, new_target) and values [total_weight, count].
agg_edges = {}

# Iterate over each edge in the original graph.
for u, v, data in token_subgraph.edges(data=True):
    # Remap u and v: if the node is part of the subject or answer group, replace it.
    new_u = aggregated_subject if u in subject_nodes else (aggregated_answer if u in answer_nodes else u)
    new_v = aggregated_subject if v in subject_nodes else (aggregated_answer if v in answer_nodes else v)
    
    key = (new_u, new_v)
    if key in agg_edges:
        agg_edges[key][0] += data['weight']
        agg_edges[key][1] += 1
    else:
        agg_edges[key] = [data['weight'], 1]

# Add the aggregated edges to the new graph with the average weight.
for (src, dst), (total_weight, count) in agg_edges.items():
    avg_weight = total_weight / count
    aggregated_graph.add_edge(src, dst, weight=avg_weight)

# For demonstration, print the aggregated graph's edges.
print("Aggregated graph edges:")
for edge in aggregated_graph.edges(data=True):
    print(edge)

In [ ]:
list(token_subgraph.edges().data())

In [ ]:
list(aggregated_graph.edges().data())